In [ ]:
import ROOT
import os
import sys
import uproot
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import math

dir = "/Users/alexanderantonakis/Desktop/Software/AFrameAnalysis/Macros/"

sys.path.append("../Utils")
sys.path.append("../Configs")
sys.path.append("../Data")

from ChannelMap import ChannelMap


os.system("ls ../Data/")


In [ ]:
frame = 0
filename = "../Data/outtree_run5328_5337_5343_5307.root"

config = "config_frame0.txt"
run_config = "run_config_frame0.txt"

fig_dir = "../Figs/RawData/Frame"+str(frame)+"/"

# set up the geometry of the frame
map = ChannelMap("../Configs/"+config, "../Configs/"+run_config)
map.initialize_config()
map.initialize_run_config()
map.calculate_params()
print("initialized the geometry and voltages")
print("")

# initialize the horizontal febs --> useful to have
febs = map.mac5
horiz_febs = []
for feb in febs:
    if map.is_horiz(feb):
        horiz_febs.append(feb)
        
print("All Horizontal FEBs in this file:", horiz_febs)

In [ ]:
# Open the ROOT file and the TTree
file = uproot.open(filename)  # Replace with your ROOT file
tree = file["reco/events"]    

run_data = tree["fRun"].array(library="np")


def get_feb_hits(row):
    f = row["flags"]
    m = row["mac5"]
    d= np.array(f) % 3 + np.array(m)
    d = list(d)
    counts = []
    for feb in horiz_febs:
        counts.append(d.count(feb))
    return counts

def get_febs(row):
    return list(set(list(row["mac5"])))

branches = ["fRun", "flags", "mac5"]  # Add the branches you want

N_vals = [0 for num in range(len(horiz_febs))]

N_vals_all = []

all_febs = []

count = 0
for run in map.runs:
    mask = (run_data == run)
    df = tree.arrays(branches, entry_start=mask.nonzero()[0][0], entry_stop=mask.nonzero()[0][-1] + 1, library="pd")
    df["feb_counts"] = df.apply(get_feb_hits, axis=1)

    df["febs"] = df.apply(get_febs, axis=1)
    for l in df["febs"].values:
        all_febs += l
    
    # Make DataFrames that are easier to work with
    hit_df = pd.DataFrame(df['feb_counts'].tolist())
    columns = []
    for feb in horiz_febs:
        columns.append("FEB"+str(feb))
    hit_df.columns = columns
    
    for num in range(len(horiz_febs)):
        #h_list[num].Fill(sumlist(hit_df["FEB"+str(horiz_febs[num])].values)))
        N_vals[num] = sum(list(hit_df["FEB"+str(horiz_febs[num])].values))
        
    N_vals = np.array(N_vals)
    N_vals_all.append(N_vals)
    N_vals = np.zeros_like(N_vals)
    count += 1

print("Hits in each run")
for arr in N_vals_all:
    print(arr)

all_febs = list(set(all_febs))
print("All febs", all_febs)

In [ ]:
kjnkjnkjnkjnkjn

In [ ]:
# Open the ROOT file and the TTree
file = uproot.open(filename)  # Replace with your ROOT file
tree = file["reco/events"]    

# Convert the TTree to a pandas DataFrame
df = tree.arrays(library="pd")

In [ ]:
df[:2]

In [ ]:
all_febs = df["mac5"].values
new_febs = []
for l in all_febs:
    febs = list(set(l))
    new_febs += febs
    
all_febs = list(set(new_febs))    
print(all_febs)

In [ ]:
all_runs = list(set(list(df["fRun"].values)))
print(all_runs)

In [ ]:
# let's check the time normalization of each run

N_vals = []
for run in all_runs:
    N = len(list(df.query("fRun == "+str(run))["fEvent"].values))
    N_vals.append(N)

h_norms = [ROOT.TH1D("h_norm_"+str(run), "", 1, 0, 1) for run in all_runs]

for num in range(len(all_runs)):
    h_norms[num].SetBinContent(1, N_vals[num])

c = ROOT.TCanvas("c", "c", 700, 500)
h_norms[0].SetLineColor(1)
h_norms[0].SetStats(0)
h_norms[0].GetXaxis().SetBinLabel(1, "")
h_norms[0].Draw("HIST")
for num in range(1, len(h_norms)):
    h_norms[num].SetLineColor(num + 1)
    h_norms[num].Draw("HIST Same")

c.Draw()

In [ ]:
def hit_count(row):
    f = row["flags"]
    return f.count(3)
    
df["N"] = df.apply(hit_count, axis=1)
df[:2]

In [ ]:


run_hits = []
for run in all_runs:
    run_hits.append(sum(list(df.query("fRun == "+str(run))["N"].values)))

h_hits = [ROOT.TH1D("h_hits_"+str(run), "", 1, 0, 1) for run in all_runs]

for num in range(len(run_hits)):
    h_hits[num].SetBinContent(1, run_hits[num])

h_rates = [h_hits[num].Clone("rate_"+str(all_runs[num])) for num in range(len(all_runs))]

for num in range(len(h_rates)):
    h_rates[num].Scale(1.0/h_norms[num].GetBinContent(1))


run_rates = np.array(run_hits) / np.array(N_vals)
run_rates = list(run_rates)


c = ROOT.TCanvas("c", "c", 700, 500)
#c.SetLogy()
h_rates[0].GetYaxis().SetRangeUser(min(run_rates) - 1, max(run_rates)+ 3)
h_rates[0].SetLineColor(1)
h_rates[0].SetLineWidth(2)
h_rates[0].SetStats(0)
h_rates[0].GetXaxis().SetBinLabel(1, "")
h_rates[0].GetYaxis().SetTitle("Average Rate [per Event]")
h_rates[0].Draw("HIST")
leg = ROOT.TLegend(0.7, 0.6, 0.9, 0.87)
leg.SetBorderSize(0)
leg.AddEntry(h_rates[0], "Run "+str(all_runs[0]))
for num in range(1, len(h_rates)):
    h_rates[num].SetLineColor(num + 1)
    h_rates[num].SetLineWidth(2)
    h_rates[num].Draw("HIST Same")
    leg.AddEntry(h_rates[num], "Run "+str(all_runs[num]))
    
leg.Draw("Same")

c.Draw()